In [ ]:
# Run batches of CoCiP on slurm
import subprocess
import time
bash_path = "/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/CoCiP/run_CoCiP_batches.sh"

EInvpm_values = ["5e12", "5e13", "5e14", "5e15"]
r_ice_values = ["0.5e-6", "1e-6", "1.5e-6", "2e-6", "3e-6"]

for i in range(len(EInvpm_values)):
    make_new_csv = True  # Set to True to create a new CSV file for the flight. A new CSV is needed if you change the flight properties (ex. EInvpm or wingspan)
    for j in range(len(r_ice_values)):
        EInvpm_string = EInvpm_values[i]
        r_ice_string = r_ice_values[j]

        arg1 = EInvpm_string
        arg2 = r_ice_string

        export_args = f"ARG1={arg1},ARG2={arg2},ARG3={make_new_csv}"

        # Update where the slurm output file is saved to
        with open(bash_path, "r") as file:
            bash_lines = file.readlines()

        # Modify the output file path in the bash script
        for j, line in enumerate(bash_lines):
            if "#SBATCH -o" in line:
                bash_lines[j] = f"#SBATCH -o /home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/CoCiP/micro_sweeps_110_218/EInvpm/{EInvpm_string}/{r_ice_string}-slurm-%j-out\n"
                break

        # Write the modified bash script back to the file
        with open(bash_path, "w") as file:
            file.writelines(bash_lines)

        # Submit the job and get the job ID
        subprocess.run(
        ["sbatch", f"--export={export_args}", bash_path], check=True
        )

        time.sleep(5)  # Sleep for 1 second between submissions to avoid overloading the scheduler
        make_new_csv = False  # Only make a new CSV for the first instance of each EInvpm


Submitted batch job 529119
